### Integration vectordb context pipeline with LLM output

In [12]:
# simple RAG pipeline using Groq API
from langchain_groq import ChatGroq
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from pathlib import Path
from typing import Any, Dict, List
import chromadb
import numpy as np
import os
from dotenv import load_dotenv

for dotenv_path in (Path(".env"), Path("../.env")):
    if dotenv_path.exists():
        load_dotenv(dotenv_path=dotenv_path)
        break

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY is not set. Add it to your .env file before running this cell.")


llm = ChatGroq(api_key=groq_api_key, model_name="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)


def get_vector_store_path():
    for path in (Path("data/vector_store"), Path("../data/vector_store")):
        if path.exists():
            return str(path)
    raise FileNotFoundError("Could not find data/vector_store. Run the vector store setup first.")


class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(texts, show_progress_bar=False)


class VectorStoreLoader:
    def __init__(self, collection_name="documents", persist_directory=None):
        self.persist_directory = persist_directory or get_vector_store_path()
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_collection(name=collection_name)


def get_document_text(doc):
    if isinstance(doc, dict):
        return doc.get("document") or doc.get("text") or doc.get("content") or ""
    return getattr(doc, "page_content", str(doc))


def set_document_text(doc, text):
    if isinstance(doc, dict):
        doc["document"] = text
    else:
        doc.page_content = text
    return doc


def get_document_metadata(doc):
    if isinstance(doc, dict):
        return doc.get("metadata") or {}
    return getattr(doc, "metadata", {}) or {}


def get_page_number(doc):
    page = get_document_metadata(doc).get("page", 0)
    try:
        return int(page)
    except (TypeError, ValueError):
        return 0


def deduplicate_docs_advanced(docs):
    seen_text = set()
    seen_meta = set()
    unique_docs = []

    for doc in docs:
        text = " ".join(get_document_text(doc).split())
        metadata = get_document_metadata(doc)
        meta_key = (metadata.get("source") or metadata.get("source_file"), metadata.get("page"))

        if text not in seen_text and meta_key not in seen_meta:
            seen_text.add(text)
            seen_meta.add(meta_key)
            unique_docs.append(set_document_text(doc, text))

    return unique_docs


def clean_docs(docs):
    return deduplicate_docs_advanced(docs)


def filter_attention_docs(docs):
    relevant_docs = []

    for doc in docs:
        text = get_document_text(doc).lower()

        # keep only attention-related content
        if "attention" in text:
            relevant_docs.append(doc)

    return relevant_docs if len(relevant_docs) > 0 else docs


def get_source_names(docs):
    sources = []
    for doc in docs:
        metadata = get_document_metadata(doc)
        source = metadata.get("source_file", metadata.get("source", ""))
        if source and source not in sources:
            sources.append(source)
    return sources


def compress_context(docs):
    docs = sorted(deduplicate_docs_advanced(docs), key=get_page_number)
    docs = docs[:6]

    seen = set()
    final_docs = []

    for doc in docs:
        text = get_document_text(doc).strip()
        if text not in seen:
            seen.add(text)
            final_docs.append(text)

    return final_docs


class RAGRetriever:
    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        self.keyword_docs = self._load_keyword_docs()
        self.bm25 = self._build_bm25(self.keyword_docs)
        self.reranker = None

    def _load_keyword_docs(self) -> List[Dict[str, Any]]:
        stored = self.vector_store.collection.get(include=["documents", "metadatas"])
        docs = []
        for rank, (doc_id, document, metadata) in enumerate(
            zip(stored.get("ids", []), stored.get("documents", []), stored.get("metadatas", [])),
            start=1,
        ):
            docs.append({
                "id": doc_id,
                "document": document,
                "metadata": dict(metadata or {}),
                "similarity_score": None,
                "distance": None,
                "rank": rank,
            })
        return clean_docs(docs)

    def _build_bm25(self, docs):
        corpus = [get_document_text(doc) for doc in docs]
        tokenized_corpus = [doc.split() for doc in corpus]
        return BM25Okapi(tokenized_corpus) if tokenized_corpus else None

    def bm25_search(self, query, docs=None, k=6):
        docs = docs or self.keyword_docs
        if not docs or self.bm25 is None:
            return []

        tokenized_query = query.split()
        scores = self.bm25.get_scores(tokenized_query)
        ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
        return [doc.copy() if isinstance(doc, dict) else doc for _, doc in ranked[:k]]

    def _get_reranker(self):
        if self.reranker is None:
            self.reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        return self.reranker

    def _rerank(self, query, docs, k=6):
        if not docs:
            return []

        try:
            pairs = [(query, get_document_text(doc)) for doc in docs]
            scores = self._get_reranker().predict(pairs)
            ranked_docs = [doc for _, doc in sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)]
            return sorted(ranked_docs[:k], key=get_page_number)
        except Exception as exc:
            print(f"Reranker unavailable, using hybrid retrieval order: {exc}")
            return sorted(docs[:k], key=get_page_number)

    def _mmr_select(self, query_embedding: np.ndarray, candidate_embeddings: List[List[float]], k: int, lambda_mult: float = 0.5) -> List[int]:
        if candidate_embeddings is None or len(candidate_embeddings) == 0:
            return []

        candidates = np.asarray(candidate_embeddings, dtype=np.float32)
        query_vector = np.asarray(query_embedding, dtype=np.float32).reshape(-1)

        candidate_norms = np.linalg.norm(candidates, axis=1, keepdims=True)
        query_norm = np.linalg.norm(query_vector)
        candidate_norms[candidate_norms == 0] = 1.0
        if query_norm == 0:
            query_norm = 1.0

        normalized_candidates = candidates / candidate_norms
        normalized_query = query_vector / query_norm
        query_scores = normalized_candidates @ normalized_query

        selected = []
        remaining = list(range(len(candidates)))
        while remaining and len(selected) < k:
            if not selected:
                next_idx = max(remaining, key=lambda idx: query_scores[idx])
            else:
                selected_vectors = normalized_candidates[selected]
                next_idx = max(
                    remaining,
                    key=lambda idx: lambda_mult * query_scores[idx] - (1 - lambda_mult) * np.max(selected_vectors @ normalized_candidates[idx]),
                )
            selected.append(next_idx)
            remaining.remove(next_idx)
        return selected

    def _vector_search(self, query: str, top_k: int = 6, fetch_k: int = 20) -> List[Dict[str, Any]]:
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=max(fetch_k, top_k),
            include=["documents", "metadatas", "distances", "embeddings"],
        )

        documents = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]
        distances = results.get("distances", [[]])[0]
        ids = results.get("ids", [[]])[0]
        candidate_embeddings = results.get("embeddings", [[]])[0]
        selected_indexes = self._mmr_select(query_embedding, candidate_embeddings, top_k, lambda_mult=0.5) or list(range(min(top_k, len(documents))))

        vector_docs = []
        for rank, index in enumerate(selected_indexes, start=1):
            distance = distances[index]
            similarity_score = 1 / (1 + distance)
            metadata = dict(metadatas[index] or {})
            metadata["score"] = similarity_score
            vector_docs.append({
                "id": ids[index],
                "document": documents[index],
                "metadata": metadata,
                "similarity_score": similarity_score,
                "distance": distance,
                "rank": rank,
            })
        return vector_docs

    def retrieve(self, query: str, top_k: int = 6, score_threshold: float | None = None) -> List[Dict[str, Any]]:
        vector_docs = self._vector_search(query, top_k=top_k, fetch_k=20)
        keyword_docs = self.bm25_search(query, self.keyword_docs, k=6)
        combined_docs = clean_docs(vector_docs + keyword_docs)

        if len(combined_docs) == 0:
            combined_docs = clean_docs(self._vector_search(query, top_k=top_k, fetch_k=20))

        docs = self._rerank(query, combined_docs, k=top_k)
        docs = sorted(deduplicate_docs_advanced(docs), key=get_page_number)
        return docs[:6]


# simple RAG function
def rag_simple(query, retriever, llm, top_k=6):
    docs = retriever.retrieve(query, top_k=top_k)
    if len(docs) == 0:
        docs = retriever.retrieve(query, top_k=top_k)

    docs = filter_attention_docs(docs)
    context_list = compress_context(docs)
    context = "\n\n".join(context_list)
    if not context:
        return "Not found in context"
    
    prompt = f"""
You are an expert AI assistant.

Answer ONLY using the provided context.

IMPORTANT:
- Do not add any information not present in the context
- Do not introduce external concepts
- If unsure, skip that point

Context:
{context}

Question:
{query}

Answer:
"""
    
    response = llm.invoke(prompt)
    if hasattr(response, "content"):
        response = response.content

    sources_used = get_source_names(docs)
    answer = f"""
{response.strip()}

Sources used:
- {', '.join(sources_used)}
""".strip() if sources_used else response.strip()
    return answer


In [13]:
active_retriever = globals().get("rag_retriever", globals().get("retriever"))
if active_retriever is None:
    embedding_manager = EmbeddingManager()
    vector_store = VectorStoreLoader()
    active_retriever = RAGRetriever(vector_store, embedding_manager)

answer = rag_simple("What is the attention mechanism?", active_retriever, llm)
print("Answer:", answer)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7923.94it/s]
d:\Data Sciece Mastery\RAG Learnings\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ayush\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100

Answer: The attention mechanism is a technique used in neural networks that allows models to focus on important parts of input data.


### Enhanced RAG Pipeline Features ...

In [ ]:
# --- Enhanced RAG Pipeline features ----
def rag_advanced(query, retriever, llm, top_k=6, min_score=None, return_context=False):
    """
    RAG pipeline with enhanced features:
    - Returns answer , source, confidence score, and optionally full context.
    """
    docs = retriever.retrieve(query, top_k=top_k)
    
    if len(docs) == 0:
        docs = retriever.retrieve(query, top_k=top_k)
    if not docs:
        return {
            "answer": "Not found in context",
            "sources": [],
            "confidence_score": 0.0,
            "context": ""
        }
    
    docs = filter_attention_docs(docs)
    context_list = compress_context(docs)
    context = "\n\n".join(context_list)
    sources = [{
        "source": doc['metadata'].get('source_file',doc["metadata"].get('source', "unknown")),
        "page": doc['metadata'].get('page', "unknown"),
        "score": doc.get('similarity_score'),
        "preview": doc['document'][:300] + "..."
    }for doc in docs]
    scored_docs = [doc['similarity_score'] for doc in docs if doc.get('similarity_score') is not None]
    confidence = max(scored_docs) if scored_docs else 0.0

    prompt = f"""
You are an expert AI assistant.

Answer ONLY using the provided context.

IMPORTANT:
- Do not add any information not present in the context
- Do not introduce external concepts
- If unsure, skip that point

Context:
{context}

Question:
{query}

Answer:
"""
    response = llm.invoke(prompt)
    if hasattr(response, "content"):
        response = response.content

    sources_used = get_source_names(docs)
    answer = f"""
{response.strip()}

Sources used:
- {', '.join(sources_used)}
""".strip() if sources_used else response.strip()
    output = {
        "answer": answer,
        "sources": sources,
        "confidence_score": confidence
    }
    if return_context:
        output["context"] = context
    return output

active_retriever = globals().get("rag_retriever", globals().get("active_retriever", globals().get("retriever")))
if active_retriever is None:
    embedding_manager = EmbeddingManager()
    vector_store = VectorStoreLoader()
    active_retriever = RAGRetriever(vector_store, embedding_manager)

result = rag_advanced("What is the advantage of using attention mechanisms?", active_retriever, llm, top_k=6, return_context=True)
print("Answer:", result["answer"].strip())
print("Sources:", result["sources"])
print("Confidence Score:", result["confidence_score"])
print("context preview:", result["context"][:300])

Answer: Attention provides a solution by allowing the model to dynamically focus on different parts of the input sequence during processing. It helps models capture relationships between words or features more effectively.
Sources: [{'source': 'attention.pdf', 'page': 0, 'score': 0.6209468582999155, 'preview': 'Attention Mechanism The attention mechanism is a technique used in neural networks that allows models to focus on important parts of input data. It is widely used in Natural Language Processing and computer vision tasks. Instead of processing all information equally, attention assigns weights to dif...'}, {'source': 'Attention_2000.pdf', 'page': 0, 'score': 0.5088967362719161, 'preview': 'the process. Attention provides a solution by allowing the model to dynamically focus on different parts of the input sequence during processing. At its core, attention works by computing relationships between elements of a sequence. It involves three main components: queries, keys, and values.

In [19]:
# advaced RAG pipeline: streaming, citation, historu, summarization, follow-up questions, etc.
from typing import List, Dict, Any

if "get_document_text" not in globals():
    def get_document_text(doc):
        if isinstance(doc, dict):
            return doc.get("document") or doc.get("text") or doc.get("content") or ""
        return getattr(doc, "page_content", str(doc))

if "get_document_metadata" not in globals():
    def get_document_metadata(doc):
        if isinstance(doc, dict):
            return doc.get("metadata") or {}
        return getattr(doc, "metadata", {}) or {}

if "get_page_number" not in globals():
    def get_page_number(doc):
        page = get_document_metadata(doc).get("page", 0)
        try:
            return int(page)
        except (TypeError, ValueError):
            return 0

if "set_document_text" not in globals():
    def set_document_text(doc, text):
        if isinstance(doc, dict):
            doc["document"] = text
        else:
            doc.page_content = text
        return doc

if "deduplicate_docs_advanced" not in globals():
    def deduplicate_docs_advanced(docs):
        seen_text = set()
        seen_meta = set()
        unique_docs = []

        for doc in docs:
            text = " ".join(get_document_text(doc).split())
            metadata = get_document_metadata(doc)
            meta_key = (metadata.get("source") or metadata.get("source_file"), metadata.get("page"))

            if text not in seen_text and meta_key not in seen_meta:
                seen_text.add(text)
                seen_meta.add(meta_key)
                unique_docs.append(set_document_text(doc, text))

        return unique_docs

if "compress_context" not in globals():
    def compress_context(docs):
        docs = sorted(deduplicate_docs_advanced(docs), key=get_page_number)
        docs = docs[:6]
        seen = set()
        final_docs = []

        for doc in docs:
            text = get_document_text(doc).strip()
            if text not in seen:
                seen.add(text)
                final_docs.append(text)

        return final_docs

if "filter_attention_docs" not in globals():
    def filter_attention_docs(docs):
        relevant_docs = []

        for doc in docs:
            text = get_document_text(doc).lower()

            # keep only attention-related content
            if "attention" in text:
                relevant_docs.append(doc)

        return relevant_docs if len(relevant_docs) > 0 else docs

if "get_source_names" not in globals():
    def get_source_names(docs):
        sources = []
        for doc in docs:
            metadata = get_document_metadata(doc)
            source = metadata.get("source_file", metadata.get("source", ""))
            if source and source not in sources:
                sources.append(source)
        return sources

class AdavancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []
    
    def query(self, question: str, top_k: int = 6, min_score=None, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        docs = self.retriever.retrieve(question, top_k=top_k)
        if len(docs) == 0:
            docs = self.retriever.retrieve(question, top_k=top_k)
        if not docs:
            answer = "Not found in context"
            sources = []
            context = ""
            summary = None
            self.history.append({
                "question": question,
                "answer": answer,
                "sources": sources,
                "summary": summary
            })
            return {
                "question": question,
                "answer": answer,
                "sources": sources,
                "summary": summary,
                "history": self.history
            }

        docs = filter_attention_docs(docs)
        context_list = compress_context(docs)
        context = "\n\n".join(context_list)
        sources = [{
            "source": doc['metadata'].get('source_file', doc["metadata"].get('source', "unknown")),
            "page": doc['metadata'].get('page', "unknown"),
            "score": doc.get('similarity_score'),
            "preview": get_document_text(doc)[:300] + "..."
        } for doc in docs]

        prompt = f"""
You are an expert AI assistant.

Answer ONLY using the provided context.

IMPORTANT:
- Do not add any information not present in the context
- Do not introduce external concepts
- If unsure, skip that point

Context:
{context}

Question:
{question}

Answer:
"""
        if stream:
            print("Generating answer...")

        response = self.llm.invoke(prompt)
        if hasattr(response, "content"):
            response = response.content

        answer = response.strip()
        sources_used = get_source_names(docs)
        answer_with_citation = f"""
{answer}

Sources used:
- {', '.join(sources_used)}
""".strip() if sources_used else answer

        summary = None
        if summarize:
            summary_prompt = f"""Summarize the following answer concisely:\n\n{answer_with_citation}\n\nSummary:"""
            summary_response = self.llm.invoke(summary_prompt)
            summary = summary_response.content if hasattr(summary_response, "content") else summary_response
            summary = summary.strip()

        self.history.append({
            "question": question,
            "answer": answer,
            "sources": sources,
            "summary": summary
        })

        return {
            "question": question,
            "answer": answer_with_citation,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }

adv_rag = AdavancedRAGPipeline(active_retriever, llm)
result = adv_rag.query("What are the benefits of attention mechanisms in deep learning?", top_k=6, stream=True, summarize=True)
print("Final Answer:", result["answer"].strip())
print("summary:", result["summary"])
print("history:", result["history"][-1])


Generating answer...
Final Answer: The benefits of attention mechanisms in deep learning include:

1. Efficient processing of long sequences
2. Ability to dynamically focus on different parts of the input sequence during processing
3. Capturing relationships between elements of a sequence
4. Assigning weights to different parts of the input to capture relationships between words or features more effectively
5. Improved interpretability by examining attention weights to understand which parts of the input the model considers important
6. Ability to capture different types of relationships simultaneously, such as syntactic and semantic connections.

Sources used:
- Attention_2000.pdf, attention.pdf, LLMs_2000.pdf
summary: The benefits of attention mechanisms in deep learning include:

1. Efficient processing of long sequences
2. Dynamic focus on input sequence parts
3. Capturing relationships between sequence elements
4. Assigning weights to input parts for effective relationships captur